# Week 4 — The ReAct Loop & Tool-Use

> **Source notebook for** [`src/react_agent/`](../src/react_agent/).

## Learning objectives

1. Formalize the ReAct paradigm (Yao et al., 2023) as a partially-observable decision process over an LLM policy.
2. Implement the Thought → Action → Observation loop in pure Python with no agent framework.
3. Design a robust text protocol parser that recovers from malformed LLM output.
4. Articulate and mitigate the four canonical failure modes: parse errors, action hallucination, tool exceptions, and infinite loops.
5. Build a tool registry whose JSON-Schema is the source of truth for what the agent can do.


## 1. ReAct as a control loop

A ReAct agent is the closed-loop interaction between an LLM policy $\pi_\theta$ and an environment of tools $\mathcal{T} = \{T_1, \dots, T_n\}$, mediated by a deterministic parser. At time $t$ the agent's state is the conversation history $h_t$. The policy emits a structured action

$$
a_t = \pi_\theta(h_t) \in \mathcal{A} = \mathcal{T} \times \mathcal{D} \;\cup\; \{\text{HALT}\},
$$

where $\mathcal{D}$ is the (tool-specific) JSON-Schema-validated input domain. If $a_t \neq \text{HALT}$, the environment returns an observation $o_t = T_a(d_a)$ and the history is updated:

$$
h_{t+1} = h_t \,\Vert\, a_t \,\Vert\, o_t.
$$

The loop terminates either when $a_t = \text{HALT}$ (the model emits a final answer) or when a hard budget $T_{\max}$ is exhausted.

This is not the only way to do tool-use — native function-calling APIs (OpenAI, Anthropic) bake the schema into the inference call. We use a text protocol instead because it (a) is transparent, (b) works with any LLM, and (c) forces the implementation to handle malformed output robustly — which it must, in practice.


## 2. The text protocol

The agent prompts the model to emit traces in one of two formats:

**Tool-call step:**
```
Thought: <free-form reasoning>
Action: <tool_name>
Action Input: <JSON object matching the tool's input_schema>
```

**Terminal step:**
```
Thought: <free-form reasoning>
Final Answer: <free-form answer>
```

The parser ([`src/react_agent/parser.py`](../src/react_agent/parser.py)) recognizes both forms and produces a `Step` object the agent loop consumes.


In [ ]:
from src.react_agent.parser import parse_step, ParseError

raw_tool_call = '''Thought: I need the product of 7 and 6.
Action: calculator
Action Input: {"expression": "7 * 6"}'''

step = parse_step(raw_tool_call)
print("is_final:", step.is_final)
print("action:  ", step.action)
print("input:   ", step.action_input)


In [ ]:
raw_final = '''Thought: The product is 42.
Final Answer: 42'''

step = parse_step(raw_final)
print("is_final:    ", step.is_final)
print("final_answer:", step.final_answer)


In [ ]:
# The parser tolerates code fences around the JSON — LLMs love to add them.
forgiving = '''Thought: compute.
Action: calculator
Action Input: ```json
{"expression": "2 ** 10"}
```'''

print(parse_step(forgiving).action_input)


In [ ]:
# When parsing fails, we surface a typed error the agent can recover from.
try:
    parse_step("Thought: just thinking aloud")
except ParseError as e:
    print("parse error:", e)


## 3. Tools as schema-validated callables

A tool is the same `Tool` object we used for MCP in Week 1 — its JSON-Schema is the contract.


In [ ]:
from src.react_agent.tools import calculator, read_file

print(calculator.describe())


In [ ]:
# The validator catches malformed arguments before the handler runs.
from src.mcp_core.protocol import MCPError
try:
    calculator.invoke({"expression": 123})    # not a string
except MCPError as exc:
    print("rejected:", exc.code, exc.message)


## 4. The agent loop

The agent itself is a 100-line `while` loop:


In [ ]:
from src.react_agent.agent import ReActAgent
from src.utils.llm_client import Message

class FakeLLM:
    \"\"\"A scripted LLM that returns canned ReAct steps in order.\"\"\"
    def __init__(self, scripted): self.scripted = list(scripted)
    def complete(self, messages, **kw): return self.scripted.pop(0)

# A two-step plan: compute, then answer.
llm = FakeLLM([
    'Thought: I will compute 17 * 23.\nAction: calculator\nAction Input: {"expression": "17 * 23"}',
    'Thought: 17 × 23 = 391.\nFinal Answer: 17 * 23 = 391',
])

agent = ReActAgent(llm=llm, tools=[calculator], max_steps=5)
run = agent.run("What is 17 times 23?")

print("answer:        ", run.answer)
print("steps:         ", run.steps)
print("halted_reason: ", run.halted_reason)
print("---")
for i, entry in enumerate(run.trace, 1):
    print(f"step {i}: action={entry.action!r:14s} observation={entry.observation!r}")


## 5. The four failure modes

A robust agent handles every one of these gracefully. The implementation does.


### 5a. Parse failure

If the LLM emits malformed output, the agent injects a corrective message and retries up to `parse_retry_budget` times before halting.


In [ ]:
llm = FakeLLM([
    "this is not in the required format",                     # garbage
    "Thought: ok, parsing now.\nFinal Answer: hello",         # recovers after corrective msg
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=5, parse_retry_budget=2)
run = agent.run("ping")
print("answer:", run.answer, " | halted_reason:", run.halted_reason)


### 5b. Action hallucination

If the LLM calls a tool that does not exist, the error is surfaced as an Observation. The LLM can then correct itself.


In [ ]:
llm = FakeLLM([
    'Thought: try a ghost tool.\nAction: ghost_tool\nAction Input: {}',
    'Thought: that tool does not exist.\nFinal Answer: I had to abandon ghost_tool.',
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("hallucinate a tool")
print("step 1 observation:", run.trace[0].observation[:80], "...")
print("final answer:      ", run.answer)


### 5c. Tool exception

If the handler raises, the exception is converted to an Observation. The agent can adapt.


In [ ]:
llm = FakeLLM([
    'Thought: divide by zero.\nAction: calculator\nAction Input: {"expression": "1/0"}',
    'Thought: division by zero. I will try something else.\nFinal Answer: cannot divide by zero.',
])
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("1/0")
print("step 1 observation:", run.trace[0].observation)
print("final answer:      ", run.answer)


### 5d. Infinite loops

A hard `max_steps` budget bounds the loop. The agent reports `halted_reason="max_steps"` so the caller can distinguish a real answer from a forced termination.


In [ ]:
# An LLM that never emits Final Answer.
looper = 'Thought: not yet.\nAction: calculator\nAction Input: {"expression": "1+1"}'
llm = FakeLLM([looper] * 10)
agent = ReActAgent(llm=llm, tools=[calculator], max_steps=3)
run = agent.run("loop forever")
print(run.halted_reason, "after", run.steps, "steps")


## 6. Putting it together: a real arXiv research agent

The same loop, with real tools (arXiv search + calculator), solves multi-step research tasks. The notebook does not execute the live call (it would require an API key), but the pattern is:

```python
from src.utils.llm_client import build_client
from src.react_agent.agent import ReActAgent
from src.react_agent.tools import calculator, search_arxiv

agent = ReActAgent(
    llm=build_client("anthropic", "claude-opus-4-7"),
    tools=[calculator, search_arxiv],
    max_steps=10,
)
run = agent.run(
    "Find three 2024 papers on HyDE, list their titles, "
    "and compute the average length of their abstracts in words."
)
print(run.answer)
```

The model interleaves `search_arxiv` calls to gather data with `calculator` calls to aggregate, then emits a Final Answer. Each step is a fresh LLM call; the conversation history is the only thing that ties them together.


## 7. Why no agent framework

Frameworks like LangChain and CrewAI hide the loop above behind layers of abstraction. The cost: every framework upgrade can break behaviour you rely on, and debugging requires excavating through wrappers.

The agent in this notebook is **~150 lines** of code, including the parser, the registry, and the loop. Once written, it is **stable**: no upstream dependency churn, no behavioural drift. The price you pay is that you write (and own) the loop. For systems that must run reliably in production, this is the right trade.


## 8. Exercises

1. **Trace verbosity.** Add a `verbosity` parameter that, at `level=2`, prints each Thought / Action / Observation as the loop runs. (Hint: `structlog.contextvars.bind_contextvars`.)
2. **Constrained decoding.** Replace the text-protocol parser with a JSON Schema-constrained decoder using the native tool-use API. Measure the parse-error rate.
3. **Streaming Observations.** What if a tool produces a 50 MB result? Modify the agent to truncate observations to the first $K$ tokens and store the full result on disk, exposing a read-tail tool to fetch more.
4. **Self-consistency.** Run the agent $K$ times at temperature $T > 0$ and majority-vote on the final answer. On a 20-question math benchmark, plot accuracy vs. $K$.


## 9. Take-aways

- ReAct is a *control loop*, not a library. The implementation fits on one screen.
- The **schema is the contract** between the LLM and the tool. Validate before invoking.
- Plan for the failure modes — parse, hallucination, exception, infinite loop — at the start. Each has a specific, recoverable mitigation.
- Native function-calling APIs are a fine optimization, but the text protocol is the right substrate for *understanding* and *debugging* what your agent is doing.

➡ Next week we move from a single agent to a *system* of specialized agents collaborating with state and memory.
